# env

In [ ]:
///- --test static

open testing

In [ ]:
open sm'_operators

## rust

### var_error

In [ ]:
nominal var_error =
    `(
        rust_type
            `($'std_env_VarError')
            `($'std::env::VarError')
            "type std_env_VarError = class end"
    )

### get_environment_variable_comptime

In [ ]:
inl get_environment_variable_comptime (var : string) : string =
    run_target_args (fun () => var) function
        | Rust => fun var =>
            open rust.rust_operators
            !\($'"option_env\!(\\\"" + !var + "\\\").unwrap_or(\\\"\\\")"')
            |> sm'.ref_to_std_string
            |> sm'.from_std_string
        | target => fun _ => null ()

## python

### os_environ

In [ ]:
nominal os_environ = any

inl os_environ () : os_environ =
    backend_switch {
        Gleam = fun () => "env.os_environ" |> failwith : os_environ
        Lua = fun () => "env.os_environ" |> failwith : os_environ
        Python = fun () =>
            global "import os"
            $'os.environ' : os_environ
    }

inl environ_get (key : string) (os_environ : os_environ) : string =
    backend_switch {
        Gleam = fun () => "env.environ_get" |> failwith : string
        Lua = fun () => "env.environ_get" |> failwith : string
        Python = fun () =>
            $'!os_environ.get(!key)' : string
    }

## env

### get_environment_variable

In [ ]:
let get_environment_variable (var : string) : string =
    inl otherwise () =
        run_target_args (fun () => var) function
            | Gleam _ => fun var =>
                backend_switch {
                    Gleam = fun () =>
                        global "import envoy"
                        global "import gleam/result"
                }
                $'!var |> envoy.get |> result.unwrap("")' : string
            | Fsharp => fun var =>
                var
                |> $'System.Environment.GetEnvironmentVariable'
                |> optionm'.of_obj
                |> optionm'.unbox
                |> optionm'.default_value ""
            | Lua => fun var =>
                $'os.getenv(!var) or ""' : string
            | Rust => fun var =>
                $'Rc::<str>::from(std::env::var(&*!var).unwrap_or_default())' : string
            | TypeScript => fun var =>
                $'(process.env[!var] ?? "")' : string
            | Cuda _ => fun var =>
                os_environ ()
                |> environ_get var
                |> optionm'.of_obj
                |> optionm'.unbox
                |> optionm'.default_value ""
            | target => fun var =>
                join "env.get_environment_variable / target: " ++#? target ++# " / var: " ++#? var |> failwith
    backend_switch_or otherwise {
        Zig = fun () =>
            $'(std.process.Environ.getAlloc(spiral_environ, spiral_gpa, !var) catch "")' : string
    }

In [1]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

"PATH2"
|> get_environment_variable
|> sm'.length
|> _assert_eq 0i32

.gleam (gleam)
__assert_eq { expected = 0 }
.lua (lua)
__assert_eq { expected = 0 }
.fsx (fsharp)
__assert_eq { expected = 0 }
.py (cuda)
__assert_eq { expected = 0 }
.rs (rust)
__assert_eq { expected = 0 }
.ts (typescript)
__assert_eq { expected = 0 }

0

In [2]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

"PATH"
|> get_environment_variable
|> sm'.length
|> fun x =>
    if x > 0i32
    then 1
    else 0
    |> _assert_eq 1i32

.gleam (gleam)
__assert_eq { expected = 1 }
.lua (lua)
__assert_eq { expected = 1 }
.fsx (fsharp)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

### get_entry_assembly_name

In [ ]:
let get_entry_assembly_name () : string =
    run_target function
        | Rust => fun () => (join "CARGO_PKG_NAME") |> get_environment_variable
        | Fsharp _ => fun () => $'System.Reflection.Assembly.GetEntryAssembly().GetName().Name'
        | target => fun () => "env.get_entry_assembly_name / target: " ++#? target

### append_path

In [3]:
inl append_path (path : string) : option string =
    inl env_path = "PATH" |> get_environment_variable
    if env_path = ""
    then None
    else
        inl env_sep =
            if platform.is_windows ()
            then ";"
            else ":"
        path ++# env_sep ++# env_path |> Some